In [1]:
import os
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
import shutil

# Paths
RAW_IMAGES = "../data/raw/images"
RAW_ANN = "../data/raw/annotations"

OUT_IMAGES = "../data/yolo_dataset/images"
OUT_LABELS = "../data/yolo_dataset/labels"

# Create folders
for path in [
    f"{OUT_IMAGES}/train", f"{OUT_IMAGES}/val",
    f"{OUT_LABELS}/train", f"{OUT_LABELS}/val"
]:
    os.makedirs(path, exist_ok=True)

# Convert function
def convert(size, box):
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]

    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0

    w = box[1] - box[0]
    h = box[3] - box[2]

    return x * dw, y * dh, w * dw, h * dh

# Load images
images = os.listdir(RAW_IMAGES)

# Split dataset
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

def process_split(image_list, split):
    for img in image_list:
        xml_file = img.replace(".png", ".xml")

        xml_path = os.path.join(RAW_ANN, xml_file)
        img_path = os.path.join(RAW_IMAGES, img)

        if not os.path.exists(xml_path):
            continue

        tree = ET.parse(xml_path)
        root = tree.getroot()

        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)

        label_file = os.path.join(OUT_LABELS, split, img.replace(".png", ".txt"))

        with open(label_file, "w") as f:
            for obj in root.findall("object"):
                bbox = obj.find("bndbox")

                xmin = float(bbox.find("xmin").text)
                xmax = float(bbox.find("xmax").text)
                ymin = float(bbox.find("ymin").text)
                ymax = float(bbox.find("ymax").text)

                bb = convert((w, h), (xmin, xmax, ymin, ymax))

                # class_id = 0 (only one class)
                f.write(f"0 {' '.join(map(str, bb))}\n")

        # Copy image
        shutil.copy(img_path, os.path.join(OUT_IMAGES, split, img))

# Run conversion
process_split(train_imgs, "train")
process_split(val_imgs, "val")

print("✅ Conversion completed successfully!")

✅ Conversion completed successfully!
